# 实验三 · Fibonacci 与 Leibniz 级数 —— 循环携带依赖

**所属**：《并行计算技术》第五章 · OpenMP 编程　|　**难度**：⭐⭐⭐ 重点　|　**预计时长**：30–40 分钟

实验二结尾指出：满足规范形式仅说明编译器**能够**分担该循环，并不意味着该循环**可以**被安全并行。本实验由此展开。实验包含两个案例：前者的依赖无法消除，后者的依赖可以消除；案例二的六个版本将「循环携带依赖」与「共享变量数据竞争」这两个彼此独立的问题分离开来，逐一辨析。

> **实验说明**
> 1. 本实验采用**递进式的版本组织**：以串行实现为基准，每个版本仅引入一种新的 OpenMP 构造或一处相应的代码改写，并在同一次运行中完成全部版本的计时与正确性校验，因而各版本面对的是完全相同的数据与运行环境，各版本之间具有可比性。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖支持 OpenMP 的 **GCC 编译器**，建议在华为鲲鹏处理器或其他 AArch64 平台上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 本实验包含两个可执行程序：`Fibonacci`案例 演示不可消除的依赖，`π估计`案例 演示可消除的依赖。请按顺序完成，前者建立判断标准，后者给出消除手法。
> 6. `π估计`案例 的命令行参数 `n` 请取**奇数**（例如 100000001）。取偶数时 V2 可能恰好给出正确结果，第 12 节将解释这一现象。
> 7. V1、V2、V3 三个版本**预期校验失败**（显示 FAIL）。此为实验的设计意图，用以展示错误行为。V3 的失败与迭代总数有关，详见 3.4 节。

## 🎯 学习目标

完成本实验后，学生应能够：

- 给出**循环携带依赖**的定义，能够在一段循环代码中指出依赖的来源与依赖距离
- 区分**流依赖**（RAW）、**反依赖**（WAR）与**输出依赖**（WAW）三种形式
- 掌握判定循环能否并行的实用方法：检查是否存在跨迭代的读写次序要求
- 理解编译器**不对被分担的循环做依赖分析**，因而「编译无告警」不能作为并行正确性的依据
- 识别两类可消除的依赖：**归纳变量**与**符号递推**，并掌握其改写手法
- 把「循环携带依赖」与「共享变量数据竞争」区分开，理解为何单独修复其中一个并不足以使程序正确
- 解释 `-O3` 优化下编译器把共享变量提升进寄存器的现象，及其对数据竞争可观测性的影响

## 🗺️ 学习路径

1. **案例一 · Fibonacci**：一个依赖无法消除的循环，观察错误如何在线程块的起始位置出现
2. **案例二 · π 估计**：一个依赖可以消除的循环，六个版本递进
3. **V0 串行基准** → **V1 共享变量 + 依赖**（双重错误）
4. **V2 `firstprivate` + 依赖**：消除了竞争，依赖仍在
5. **V3 共享变量 + 依赖已消除**：消除了依赖，竞争仍在
6. **V4 `private` + 依赖已消除**：两个问题同时消除，结果正确
7. **V5 数学展开**：改写算法本身，无分支且可向量化
8. **可视化与分析**：绘制加速比柱状图，讨论正确性与性能的先后次序

## 1. 背景知识：什么是循环携带依赖

### 1.1 定义

若循环的两次不同迭代 $i$ 与 $j$（$j \ne i$）访问同一内存位置，且其中**至少有一次为写操作**，则称该循环存在**循环携带依赖**（loop-carried dependence）。存在依赖意味着迭代之间存在先后次序要求；并行执行不再保证该次序，因而结果可能不正确。至于错误是否可复现，取决于依赖的具体形式——本实验 8.3 节的 V2 即是一个错误完全确定的例子。

考察下面这个最简单的例子：

```c
for (int i = 1; i < n; i++) {
  a[i] = a[i - 1] + 1;   // 第 i 次迭代读取第 i-1 次迭代写入的值
}
```

第 $i$ 次迭代必须在第 $i-1$ 次之后执行。若将该循环分配给四个线程执行，线程 1 从 `i = n/4` 开始时，`a[n/4 - 1]` 可能尚未被线程 0 写入，从而读到旧值。

### 1.2 依赖的三种形式

| 形式 | 全称 | 模式 | 能否通过改写消除 |
|---|---|---|---|
| **流依赖** | RAW，Read After Write | 先写后读 | 通常不能，属于真实的数据流 |
| **反依赖** | WAR，Write After Read | 先读后写 | 通常能，通过增加副本消除 |
| **输出依赖** | WAW，Write After Write | 写后再写 | 通常能，通过私有化消除 |

流依赖也称**真依赖**，因为它反映的是算法本身的数据流向；反依赖与输出依赖则称**伪依赖**，它们源于存储位置的复用而非真实的数据流向，引入额外的存储即可消除。

### 1.3 依赖距离

把「第 $i$ 次迭代依赖第 $i-d$ 次迭代」中的 $d$ 称为**依赖距离**。

- $d = 1$：相邻迭代之间的依赖，如上面的 `a[i] = a[i-1] + 1`；
- $d = 2$：Fibonacci 递推 `f[i] = f[i-1] + f[i-2]` 的依赖距离最大为 2；
- $d$ 与执行粒度的关系：若 $d \ge$ 向量宽度 $V$，则宽度为 $V$ 的向量化是安全的——同一向量批次内部的 $V$ 次迭代互不依赖，依赖只跨批次，而批次之间仍按原次序执行。但这一结论**不能**照搬到线程级并行：OpenMP 的块之间是并发执行的，只要依赖跨块就会被破坏，且 $d$ 越大，跨块的依赖比例越高（当 $d$ 不小于块长时，全部依赖均跨块）。

### 1.4 编译器对依赖的处理

> **编译器不对 `#pragma omp for` 所分担的循环做依赖分析。**

这与自动向量化（第三章）形成鲜明对照。自动向量化时，编译器必须自行证明迭代之间无依赖，证明不了就放弃向量化；而 `#pragma omp for` 是程序员下达的**命令**，编译器只负责执行。

因此，「编译无告警、运行不崩溃」并不能作为判断并行正确性的依据。本实验的 V1、V2、V3 三个版本均可无告警通过编译，运行时也不会产生任何异常，但它们的结果全部是错误的。

## 2. 依赖的判定与消除

### 2.1 实用判定方法

面对一个待并行的循环，依次检查以下三项：

| 检查项 | 具体做法 |
|---|---|
| **数组下标** | 循环体内是否出现 `a[i-1]`、`a[i+1]` 这类非 `a[i]` 的下标？ |
| **跨迭代的标量** | 是否有在循环外声明、循环内既读又写的标量？ |
| **函数调用** | 被调用的函数是否读写全局状态或静态变量？ |

只要三项中有任意一项为「是」，就必须进一步分析。

> **一种实用的自检方法**：将循环的迭代次序**完全逆转**（从 `n-1` 递减至 `0`）并串行执行。若结果与正序一致，则很可能不存在循环携带依赖；若结果不一致，则提示存在依赖。
>
> 需要注意：对浮点运算，逆序求和本身就会因舍入次序改变而产生末位差异，**这并不意味着存在依赖**。因此比较时应采用与并行校验相同的容差，只有**超出容差**的差异才提示依赖的存在。该方法不能证明无依赖，但发现依赖的能力较强，且无需额外工具。

### 2.2 两类可消除的依赖

**第一类：归纳变量**（induction variable）

```c
int j = 0;
for (int i = 0; i < n; i++) {
  j = j + 2;        // j 依赖上一次迭代
  a[i] = b[j];
}
```

`j` 的取值其实可以由 `i` 直接算出：`j = 2 * (i + 1)`。把递推改写为**闭式表达式**，依赖随之消失：

```c
#pragma omp parallel for
for (int i = 0; i < n; i++) {
  a[i] = b[2 * (i + 1)];
}
```

**第二类：符号递推**

```c
double factor = 1.0;
for (long k = 0; k < n; k++) {
  sum += factor / (2 * k + 1);
  factor = -factor;      // 依赖上一次迭代
}
```

`factor` 的取值仅取决于 $k$ 的奇偶：$k$ 为偶数时为 $+1$，为奇数时为 $-1$。改写为 `factor = (k % 2 == 0) ? 1.0 : -1.0;` 即可消除依赖。这正是本实验案例二的核心。

### 2.3 无法消除的依赖

Fibonacci 递推 `f[i] = f[i-1] + f[i-2]` 属于真实的数据流：第 $i$ 项的值确实由前两项决定。这类循环**不能**通过改写循环体来并行化。

若确需并行，只能更换算法。可选的途径有两条。其一是 Binet 闭式公式 $F_m = (\varphi^m - \psi^m)/\sqrt{5}$（其中 $\varphi = (1+\sqrt{5})/2$，$\psi = (1-\sqrt{5})/2$）——它确实给出了只依赖下标的表达式，但引入了无理数的浮点幂运算，在下标较大时精度不足以给出精确的整数结果。其二是矩阵快速幂：

$$\begin{pmatrix} F_{n+1} \\ F_n \end{pmatrix} = \begin{pmatrix} 1 & 1 \\ 1 & 0 \end{pmatrix}^n \begin{pmatrix} 1 \\ 0 \end{pmatrix}$$

但这两条途径都已属于**更换算法**，而非对原循环的并行化：前者改变了计算的数值性质，后者改变了计算的组织结构。这一区分很重要：**并行化是在保持算法不变的前提下改变执行方式；更换算法则属于不同范畴**。

## 3. OpenMP 关键知识点

### 3.1 本实验涉及的两个正交问题

并行化一个含有跨迭代标量的循环时，可能同时存在两个彼此**独立**的问题：

| 问题 | 本质 | 表现 | 修复手段 |
|---|---|---|---|
| **循环携带依赖** | 算法层面的次序要求 | 结果系统性偏离 | 改写为闭式表达 |
| **共享变量竞争** | 多线程同时读写同一内存 | 结果随运行波动 | `private` 私有化 |

初学阶段常见的误解是将两者混为一谈，以为「添加 `private` 即可并行」或「消除依赖后程序即正确」。案例二的六个版本正是为分离这一混淆而设计：

```text
                      依赖存在          依赖已消除
                  ┌───────────────┬───────────────┐
     变量共享     │  V1  双重错误 │  V3  仅剩竞争 │
                  ├───────────────┼───────────────┤
     变量私有     │  V2  仅剩依赖 │  V4  结果正确 │
                  └───────────────┴───────────────┘
```

只有两个问题同时解决（V4），程序才是正确的。

### 3.2 `private` 与 `firstprivate` 的区别

实验二已经指出 `private` 不复制初值。本实验将直接观察这一性质的后果。

```c
double factor = 1.0;

#pragma omp parallel for private(factor)        // 各副本初值未定义
#pragma omp parallel for firstprivate(factor)   // 各副本初值为 1.0
```

V2 选用 `firstprivate` 而非 `private`，是为了让错误**可复现**：

- 用 `private` 时，各线程 `factor` 副本的初值未定义（读取未初始化的变量属于未定义行为），结果每次运行都不相同，不便于分析；
- 用 `firstprivate` 时，各线程 `factor` 的初值确定为 $+1$，于是错误有了**明确的成因**：凡是块起始下标为奇数的线程，该块内各项的符号全部与正确值相反。

这体现了一种常用的调试思路：**将不确定的错误转化为可复现的确定错误，再进行分析**。

### 3.3 `-O3` 下的寄存器提升

OpenMP 的内存模型允许编译器在没有同步点的情况下，把共享变量暂存在寄存器中，直到区域结束或遇到 `flush` 才写回内存。

这一优化对 V1 有直接影响：`factor` 虽然在语法上是共享变量，但 GCC 在 `-O3` 下可能把它整个提升进各线程的私有寄存器，使 V1 的**实际行为在效果上与 V2 相同**。因此**两者可能给出完全相同的错误值**。

> 这一现象与第四章 π 估计实验中观测到的寄存器提升同源。在第四章中，`-O3` 把 `global_sum` 整个提升进寄存器，导致丢失的不是一次更新，而是**整个线程的部分和**。

V3 为了让竞争可观测，把 `factor` 声明为 `volatile`：

```c
volatile double factor = 1.0;
```

`volatile` 要求每次访问都必须实际读写内存，禁止寄存器提升，竞争窗口因而暴露出来。

> **需指出**：`volatile` 在此仅用于教学演示，它**并非**解决数据竞争的手段。`volatile` 只保证访问不被优化掉，不提供任何原子性或内存序保证。
>
> 更需注意的是：加上 `volatile` 之后，V3 **依然是一个数据竞争程序**，其行为在标准意义上仍属未定义；`volatile` 的作用只是阻止编译器把共享变量提升进寄存器，从而使竞争在本平台上变得可观测。工程中解决竞争应使用私有化、归约或同步构造。

### 3.4 V3 复现竞争的前提

V3 的竞争窗口仅有数条指令（自写入 `factor` 至读取 `factor` 之间），因此它能否被观测到，取决于两个因素：**并发程度**与**迭代总数**。

| 运行条件 | V3 的表现 |
|---|---|
| 多核，任意规模 | 立即复现，稳定 FAIL |
| 单核，$n \approx 10^8$ | 仍然复现。抢占次数足够多，累计命中窗口 |
| 单核，$n \approx 10^7$ | 通常报告 PASS，窗口未被命中 |

源码中已加入相应的运行时提示。这一现象本身就是一条重要经验：**数据竞争的可观测性同时依赖于运行平台与问题规模**。在小规模、核心数较少的环境中测试通过的多线程程序，在实际部署环境中仍可能暴露缺陷。

## 4. 环境准备与检查

本节确认三项内容：编译器是否支持 OpenMP、运行时报告的处理器数量、以及各处理器核心的最大频率是否一致。

第三项检查针对**异构多核**平台。Arm 的 big.LITTLE 架构把高性能核心与高能效核心集成在同一块芯片上，二者的频率与微架构均不相同。在这类平台上，同一段代码在不同类型的核心上执行，耗时可能相差 2 倍以上，线程数与加速比之间因而不再是简单的线性关系。

需要说明的是，最大频率不一致只是异构多核的**必要非充分**证据：同构多核平台也可能因加速频率（boost）策略或芯片分级（binning）而上报不同的 `cpuinfo_max_freq`。因此下面的检查只给出提示，确认平台是否为异构架构还需结合 `lscpu` 输出的核心型号信息。

In [ ]:
import os
import re
import subprocess
import platform

print('=' * 60)
print(' 一、平台信息')
print('=' * 60)
print('操作系统   :', platform.system(), platform.release())
print('处理器架构 :', platform.machine())
print('逻辑核心数 :', os.cpu_count())

print()
print('=' * 60)
print(' 二、编译器与 OpenMP 支持')
print('=' * 60)
gcc_ver = subprocess.run(['gcc', '--version'], capture_output=True,
                         text=True).stdout.splitlines()[0]
print('编译器     :', gcc_ver)

probe = subprocess.run('echo | gcc -fopenmp -dM -E -x c - | grep _OPENMP',
                       shell=True, capture_output=True, text=True).stdout.strip()
if probe:
    ver = int(probe.split()[-1])
    spec = {200805: '3.0', 201107: '3.1', 201307: '4.0',
            201511: '4.5', 201811: '5.0', 202011: '5.1'}.get(ver, '未知')
    print('_OPENMP    :', ver, '(对应 OpenMP %s 规范)' % spec)
    print('数组段归约 :', '支持' if ver >= 201511 else '不支持（需要 4.5 及以上）')
else:
    print('⚠️  未检测到 OpenMP 支持，请确认编译时带有 -fopenmp')

print()
print('=' * 60)
print(' 三、核心频率与异构性检查')
print('=' * 60)
freqs = []
for cpu in range(os.cpu_count() or 1):
    path = '/sys/devices/system/cpu/cpu%d/cpufreq/cpuinfo_max_freq' % cpu
    try:
        with open(path) as f:
            freqs.append((cpu, int(f.read().strip()) // 1000))
    except OSError:
        pass

if not freqs:
    print('无法读取 cpufreq 节点，跳过异构性检查。')
else:
    for cpu, mhz in freqs:
        print('  CPU%-2d 最大频率: %5d MHz' % (cpu, mhz))
    distinct = sorted(set(m for _, m in freqs))
    if len(distinct) > 1:
        print()
        print('⚠️  检测到 %d 种不同的最大频率，本平台可能为异构多核架构（如 Arm big.LITTLE）。'
              % len(distinct))
        print('    请结合 lscpu 输出的核心型号信息进一步确认。')
        print('    若确为异构平台，测速前建议执行：')
        print('      export OMP_PROC_BIND=close')
        print('      export OMP_PLACES=cores')
    else:
        print()
        print('✅ 全部核心的最大频率一致，可按同构多核平台处理。')

print()
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', '（未设置，由运行时决定）'))
print('OMP_PROC_BIND   =', os.environ.get('OMP_PROC_BIND', '（未设置）'))
print('OMP_PLACES      =', os.environ.get('OMP_PLACES', '（未设置）'))

## 5. 实验工具函数

本节定义三个贯穿全章的辅助函数，后续各实验均直接调用，不再重复说明。

| 函数 | 作用 |
|---|---|
| `compile_c(src)` | 以 `-O3 -fopenmp -Wall -Wextra` 编译指定源文件，并回显全部告警 |
| `run_c(binary, *args)` | 运行可执行文件并原样打印其标准输出 |
| `parse_table(output)` | 从程序输出的结果表中提取「方法名 / 耗时 / 加速比 / 校验」四列 |

**关于编译选项**：全章统一使用 `-O3 -fopenmp`。AArch64 平台的 NEON 属于基线指令集，无需附加 `-march` 或 `-mcpu` 选项。`-Wall -Wextra` 用于暴露数据环境声明不当引发的告警，这类告警在 OpenMP 程序中往往是并发缺陷的征兆，不应忽略。

In [ ]:
import subprocess
import re
import os

SRC_DIR = 'src_dependence'
os.makedirs(SRC_DIR, exist_ok=True)

CFLAGS = ['-O3', '-fopenmp', '-Wall', '-Wextra']


def compile_c(src, extra=('-lm',)):
    """编译单个源文件，返回可执行文件路径；编译失败时抛出异常。"""
    src_path = os.path.join(SRC_DIR, src)
    binary = os.path.join(SRC_DIR, os.path.splitext(src)[0])
    cmd = ['gcc'] + CFLAGS + ['-o', binary, src_path] + list(extra)
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError('编译失败：%s' % src)
    print('✅ 编译通过，无告警' if not proc.stderr.strip()
          else '⚠️  编译通过，但存在告警，请逐条阅读')
    return binary


def run_c(binary, *args, env=None):
    """运行可执行文件，打印并返回其标准输出。"""
    cmd = [binary] + [str(a) for a in args]
    print('$', ' '.join(cmd))
    print()
    run_env = dict(os.environ)
    if env:
        run_env.update({k: str(v) for k, v in env.items()})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print('[stderr]', proc.stderr.rstrip())
    return proc.stdout


ROW_RE = re.compile(r'^\|\s*(.+?)\s*\|\s*([0-9.]+)\s*\|\s*([0-9.]+)x\s*\|\s*(\S+)\s*\|$')


def parse_table(output):
    """解析结果表，返回 [(方法名, 耗时ms, 加速比, 校验结论), ...]。"""
    rows = []
    for line in output.splitlines():
        m = ROW_RE.match(line.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)),
                         float(m.group(3)), m.group(4)))
    return rows


print('工具函数已就绪，源码目录：', os.path.abspath(SRC_DIR))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

C_BASE, C_GOOD, C_FAIL, C_SLOW = '#7f7f7f', '#1f77b4', '#d62728', '#ff7f0e'


def plot_speedup(rows, title, figsize=(10, 5)):
    """绘制加速比柱状图。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。"""
    if not rows:
        print('未解析到结果行，请先运行上一单元格。')
        return
    names = [r[0] for r in rows]
    speeds = [r[2] for r in rows]
    colors = []
    for i, (_, _, sp, chk) in enumerate(rows):
        if i == 0:
            colors.append(C_BASE)
        elif chk == 'FAIL':
            colors.append(C_FAIL)
        elif sp < 1.0:
            colors.append(C_SLOW)
        else:
            colors.append(C_GOOD)

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(range(len(names)), speeds, color=colors,
                  edgecolor='black', linewidth=0.6, width=0.6)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Speedup vs. Serial Baseline')
    ax.set_title(title, fontsize=12, pad=12)
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True)

    for bar, (_, ms, sp, chk) in zip(bars, rows):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                '%.2fx\n%.1f ms%s' % (sp, ms, '' if chk in ('-', 'PASS') else '\nFAIL'),
                ha='center', va='bottom', fontsize=8)

    ax.set_ylim(0, max(speeds) * 1.30)
    plt.tight_layout()
    plt.show()


print('绘图函数已就绪。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。')

## 6. 案例一 · Fibonacci：无法消除的依赖

### 6.1 代码结构

串行版本：

```c
static void fibo_serial(int *f, int n) {
  f[0] = 1;
  f[1] = 1;
  for (int i = 2; i < n; i++) {
    f[i] = f[i - 1] + f[i - 2];
  }
}
```

并行版本只增加了一行制导语句，同时记录每次迭代由哪个线程执行：

```c
#pragma omp parallel for num_threads(thread_count)
  for (int i = 2; i < n; i++) {
    f[i] = f[i - 1] + f[i - 2];
    owner[i] = omp_get_thread_num();   // 记录归属线程
  }
```

`owner` 数组的作用是把「错误出现的位置」与「线程块的边界」对应起来，使依赖被破坏的位置可以直接观察到。

### 6.2 关于 `n` 的上限

源码限制 `n <= 46`。程序中 `f[0] = f[1] = 1`，故下标 $i$ 对应数列的第 $i+1$ 项。以 `int` 存储时，下标 45 处的 $F_{46} = 1836311903$ 仍在有符号 32 位整数范围内，而 $F_{47} = 2971215073$ 已超出该范围。限定上限可以保证串行参照结果精确，使得任何不一致都确实来自并行化，而非整数溢出。

### 6.3 预期观察

以 `n = 20`、4 线程运行时，迭代空间 `[2, 20)` 共 18 次迭代，默认静态划分下每个线程约 4–5 次。线程 0 从 `i = 2` 起，其结果正确（因为 `f[0]`、`f[1]` 已在循环外赋值）。其余线程则不然，它们各自块内的第一次迭代所需的前两项，由别的线程负责写入，此时往往尚未完成，读到的仍是数组的初值 0。从这个下标开始，结果就不再等于串行参照值。最先出现偏差的是哪一个线程，取决于运行时的调度。核心数充足时通常是线程 1；可用核心数少于线程数时，也可能是更靠后的线程先进入自己的块。程序输出的 `owner` 一列给出了该下标的归属线程，可据此核对。

> 需要说明的是，该并行版本对 `f[]` 的并发读写本身即构成数据竞争，其行为在标准意义上属于未定义。上述规律是特定实现与调度下的典型表现，而非语言层面的保证——若线程 0 恰好抢先完成，个别下标也可能得到正确的值。

### 6.4 源代码写入

In [ ]:
%%writefile {SRC_DIR}/omp_fibo_dependency.c
#define _POSIX_C_SOURCE 200809L

#include <omp.h>
#include <stdio.h>
#include <stdlib.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

// fibo[45] is the largest term that still fits into a signed 32-bit integer,
// so the serial reference stays exact.
#define MAX_N 46
#define MAX_THREADS 16
#define PRINT_LIMIT 15

#define BANNER "============================================================"

// ============================================================================
// Serial reference. Iteration i must observe the results of i-1 and i-2.
// ============================================================================
static void fibo_serial(int *f, int n) {
  f[0] = 1;
  f[1] = 1;
  for (int i = 2; i < n; i++) {
    f[i] = f[i - 1] + f[i - 2];
  }
}

// ============================================================================
// Buggy parallel version. The loop carries a Read-After-Write dependence:
// a thread that starts at index i may read f[i-1] and f[i-2] before the
// thread that owns those indices has produced them.
// ============================================================================
static void fibo_parallel_wrong(int *f, int n, int thread_count, int *owner) {
  f[0] = 1;
  f[1] = 1;

#pragma omp parallel for num_threads(thread_count)
  for (int i = 2; i < n; i++) {
    f[i] = f[i - 1] + f[i - 2];
    owner[i] = omp_get_thread_num();
  }
}

int main(int argc, char *argv[]) {
  if (argc != 3) {
    printf("Usage: %s <n> <thread_count>\n", argv[0]);
    printf("       2 <= n <= %d, 1 <= thread_count <= %d\n", MAX_N,
           MAX_THREADS);
    return 1;
  }

  int n = (int)strtol(argv[1], NULL, 10);
  int thread_count = (int)strtol(argv[2], NULL, 10);

  if (n < 2 || n > MAX_N) {
    printf("Error: n must be between 2 and %d (int overflow beyond that)\n",
           MAX_N);
    return 1;
  }
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }

  int *f_ref = (int *)calloc((size_t)n, sizeof(int));
  int *f_omp = (int *)calloc((size_t)n, sizeof(int));
  int *owner = (int *)calloc((size_t)n, sizeof(int));
  if (f_ref == NULL || f_omp == NULL || owner == NULL) {
    printf("Error: memory allocation failed\n");
    return 1;
  }
  for (int i = 0; i < n; i++) {
    owner[i] = -1;
  }

  printf("%s\n", BANNER);
  printf(" Lab 3 (part 1): Loop-Carried Dependence (Fibonacci)\n");
  printf(" n: %d | Threads: %d | _OPENMP: %d\n", n, thread_count, _OPENMP);
  printf("%s\n", BANNER);

  fibo_serial(f_ref, n);
  fibo_parallel_wrong(f_omp, n, thread_count, owner);

  int limit = (n < PRINT_LIMIT) ? n : PRINT_LIMIT;
  printf("\n idx |     serial |   parallel | owner\n");
  printf("-----|------------|------------|-------\n");
  for (int i = 0; i < limit; i++) {
    printf(" %3d | %10d | %10d | %5d\n", i, f_ref[i], f_omp[i], owner[i]);
  }

  int first_bad = -1;
  for (int i = 0; i < n; i++) {
    if (f_ref[i] != f_omp[i]) {
      first_bad = i;
      break;
    }
  }

  printf("\nResult: ");
  if (first_bad < 0) {
    printf("no difference detected with %d thread(s).\n", thread_count);
    printf("        Run again with more threads: a single thread executes\n");
    printf("        the iterations in order, which hides the dependence.\n");
  } else {
    printf("first mismatch at index %d (serial %d, parallel %d).\n", first_bad,
           f_ref[first_bad], f_omp[first_bad]);
    printf("        Index %d is the first iteration of a chunk owned by\n",
           first_bad);
    printf("        thread %d, whose predecessors were not ready yet.\n",
           owner[first_bad]);
  }

  free(f_ref);
  free(f_omp);
  free(owner);
  return 0;
}

### 6.5 编译与运行

In [ ]:
bin_fibo = compile_c('omp_fibo_dependency.c', extra=())
print()
out_fibo = run_c(bin_fibo, 20, 4)

### 6.6 线程数对错误位置的影响

改变线程数会改变块的边界，因而也会改变首个错误出现的位置。下面依次以 1、2、4、8 线程运行，观察这一对应关系。

特别注意 **1 线程**的情形：单线程按顺序执行全部迭代，依赖得以满足，结果正确。这说明该程序的缺陷**只在并行时显现**，串行测试无法发现它。

In [ ]:
import re

for nt in (1, 2, 4, 8):
    out = subprocess.run([bin_fibo, '20', str(nt)],
                         capture_output=True, text=True).stdout
    m = re.search(r'first mismatch at index (\d+)', out)
    if m:
        print('线程数 %-2d → 首个错误下标 = %s' % (nt, m.group(1)))
    else:
        print('线程数 %-2d → 未检测到错误（迭代次序未被打乱）' % nt)

## 7. 案例二 · π 估计：可以消除的依赖

### 7.1 数学背景

Gregory-Leibniz 级数：

$$\pi = 4\left(1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + \frac{1}{9} - \cdots\right) = 4\sum_{k=0}^{\infty} \frac{(-1)^k}{2k+1}$$

该级数收敛极慢：取 $n = 10^8$ 项，截断误差仍在 $10^{-8}$ 量级。因此本实验的校验以**串行结果**为参照。

### 7.2 版本设计总览

| 版本 | `factor` 的数据属性 | `factor` 的求法 | 依赖 | 竞争 | 预期 |
|---|---|---|---|---|---|
| **V0** | —（串行执行） | `factor = -factor` | — | — | 基准 |
| **V1** | 共享（默认） | `factor = -factor` | 有 | 有 | **FAIL** |
| **V2** | `firstprivate` | `factor = -factor` | 有 | 无 | **FAIL** |
| **V3** | 共享 `volatile` | 由 `k` 求出 | 无 | 有 | **FAIL**（需多核） |
| **V4** | `private` | 由 `k` 求出 | 无 | 无 | PASS |
| **V5** | —（已消去） | 数学展开消去 | 无 | 无 | PASS |

这一设计的用意是把 3.1 节的二维表**逐格加以验证**。只修复依赖（V3）或只修复竞争（V2），结果都仍然是错误的；两者同时修复（V4）才正确。

## 8. 逐版本代码讲解

### 8.1 V0 · 串行基准

```c
double sum = 0.0;
double factor = 1.0;
for (long k = 0; k < n; k++) {
  sum += factor / (double)(2 * k + 1);
  factor = -factor;          // 符号递推
}
return 4.0 * sum;
```

`factor` 在循环外声明、循环内既读又写，正是 2.1 节判定表中的第二项。

### 8.2 V1 · 共享变量 + 依赖（双重错误）

```c
double factor = 1.0;
#pragma omp parallel for num_threads(thread_count) reduction(+ : sum)
for (long k = 0; k < n; k++) {
  sum += factor / (double)(2 * k + 1);
  factor = -factor;
}
```

这是并行程序设计中最典型的错误写法之一。`sum` 已用 `reduction` 正确处理，但 `factor` 被遗漏了：它在并行区域外声明，默认为**共享**，于是多个线程同时对它读写；同时它还携带跨迭代的依赖。两个问题叠加在一起。

> **实测提示**：由于 3.3 节所述的寄存器提升，本版本在 `-O3` 下的实际行为可能与 V2 完全一致，两者给出**相同的错误值**。该现象说明「源码语义」与「优化后的实际行为」之间可能存在差异。

### 8.3 V2 · `firstprivate` + 依赖（竞争已除，依赖仍在）

```c
#pragma omp parallel for num_threads(thread_count) reduction(+ : sum) \
    firstprivate(factor)
for (long k = 0; k < n; k++) {
  sum += factor / (double)(2 * k + 1);
  factor = -factor;
}
```

`firstprivate` 让每个线程持有自己的 `factor` 副本，初值均为 $+1$。数据竞争因此消失，程序的行为不再依赖线程间的读写交错：除归约合并次序引起的末位差异外，每次运行给出相同的结果。

但结果仍然是错的。原因在于：线程 $t$ 的块从下标 $k_t$ 开始，若 $k_t$ 为**奇数**，则该处的正确符号应为 $-1$，而副本初值是 $+1$，于是**整个块的符号全部反了**。

```text
  k:      0    1    2  ...  |  25000001  25000002  ...
  正确:   +    -    +       |     -          +
  V2  :   +    -    +       |     +          -        ← 整块反号
                    线程0   |  线程1（块首为奇数）
```

> **为何要求 `n` 取奇数**：默认静态划分下，若 `n` 能被线程数整除且商为偶数，则所有块的起始下标都是偶数，V2 反而会给出**正确**结果。取奇数 `n` 则可保证至少有一个块从奇数下标开始，理由如下：若所有块的起始下标均为偶数，则每个块的长度都必为偶数，各块长度之和 `n` 也必为偶数；因此当 `n` 为奇数且线程数不小于 2 时，至少存在一个长度为奇数的块，从而至少有一个块的起始下标为奇数。
>
> 「逻辑上错误的程序在特定参数下恰好给出正确结果」这一现象，是并发缺陷难以发现的典型情形，第 12 节将专门讨论。

### 8.4 V3 · 依赖已消除，竞争仍在

```c
volatile double factor = 1.0;      // 共享，且禁止寄存器提升
#pragma omp parallel for num_threads(thread_count) reduction(+ : sum)
for (long k = 0; k < n; k++) {
  factor = (k % 2 == 0) ? 1.0 : -1.0;   // 依赖已消除
  sum += factor / (double)(2 * k + 1);
}
```

`factor` 现在完全由 `k` 决定，跨迭代的依赖已经消失。但它仍然是**共享变量**：线程 A 刚写入 $+1$，线程 B 随即写入 $-1$，线程 A 接着读到的就是 $-1$。

这是一个纯粹的数据竞争，其表现与依赖造成的错误截然不同：

| | V2（依赖） | V3（竞争） |
|---|---|---|
| 每次运行结果 | 完全相同 | 略有波动 |
| 误差量级 | 较大且系统性 | 较小且随机 |
| 单核是否复现 | 一定复现 | 通常不复现 |

### 8.5 V4 · 正确版本

```c
double factor = 0.0;
#pragma omp parallel for num_threads(thread_count) reduction(+ : sum) \
    private(factor)
for (long k = 0; k < n; k++) {
  factor = (k % 2 == 0) ? 1.0 : -1.0;
  sum += factor / (double)(2 * k + 1);
}
```

依赖已由 V3 消除，此处再把 `factor` 私有化，两个问题同时解决。

注意 `factor` 的初值写作 `0.0`。该初值实际上**不会被复制到任何私有副本中**——`private` 不复制初值，而循环体内每次迭代都会先赋值再使用。写成 `0.0` 只是为了避免「声明后未初始化」的告警，并借此提醒读者：`private` 副本的初值与外层变量无关。

> **更简洁的写法**：把 `double factor` 直接声明在循环体**内部**，即可自动获得私有属性，无需任何子句。此处保留 `private` 子句是为了配合教材的讲解次序。

### 8.6 V5 · 数学展开

把相邻两项合并：

$$\frac{1}{4k+1} - \frac{1}{4k+3}$$

```c
long pairs = n / 2;
#pragma omp parallel for num_threads(thread_count) reduction(+ : sum)
for (long k = 0; k < pairs; k++) {
  sum += 1.0 / (double)(4 * k + 1) - 1.0 / (double)(4 * k + 3);
}
if (n % 2 != 0) {
  sum += 1.0 / (double)(2 * n - 1);   // n 为奇数时未配对的剩余项
}
```

改写之后，符号变量彻底消失，循环体内不再有取模运算与条件分支。这带来三重收益：

1. 依赖与竞争都不复存在，正确性一目了然；
2. 循环次数减半；
3. 消除了取模运算，循环体形态更简单，有利于编译器实施向量化。

需要说明的是，V4 中的三元运算符在 `-O3` 下通常已被编译为无分支的条件选择指令（如 AArch64 的 `fcsel`），并不构成真正的分支预测开销。因此 V5 的性能优势主要来自**循环次数减半**，而非分支的消除。

> **这一版体现了一条重要的优化次序**：面对含依赖的循环，首选是回到**算法本身**寻找改写空间，而非直接施加同步构造。同步能保证正确，但代价高昂；改写算法则可能同时获得正确性与性能。

## 9. 源代码写入

下面写入案例二的完整源码，内容与课程代码目录中的 `03b_omp_pi_dependency.c` 逐字节一致。

In [ ]:
%%writefile {SRC_DIR}/omp_pi_dependency.c
#define _POSIX_C_SOURCE 200809L

#include <math.h>
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

#define NTIMES 5
#define MAX_THREADS 16
// Reordering the additions changes the result by far less than this bound,
// while every incorrect version misses it by several orders of magnitude.
#define TOL 1e-8

#define BANNER "============================================================"
#define LINE "------------------------------------------------------------"

// ----------------------------------------------------------------------------
// Common helpers
// ----------------------------------------------------------------------------
static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static int check_diff(const double *ref, const double *test, long n,
                      double tol) {
  for (long i = 0; i < n; i++) {
    if (fabs(ref[i] - test[i]) > tol) {
      return 0;
    }
  }
  return 1;
}

static void print_table_header(void) {
  printf("\n%s\n", LINE);
  printf("| %-26s | %9s | %7s | %-5s |\n", "Method", "Time(ms)", "Speedup",
         "Check");
  printf("|----------------------------|-----------|---------|-------|\n");
}

static void print_row(const char *name, double time_ms, double base_ms,
                      int check) {
  const char *status = (check < 0) ? "-" : (check ? "PASS" : "FAIL");
  double speedup = (time_ms > 0.0) ? base_ms / time_ms : 0.0;
  printf("| %-26s | %9.3f | %6.2fx | %-5s |\n", name, time_ms, speedup, status);
}

// ============================================================================
// V0: Serial baseline
// ============================================================================
static double pi_serial(long n) {
  double sum = 0.0;
  double factor = 1.0;

  for (long k = 0; k < n; k++) {
    sum += factor / (double)(2 * k + 1);
    factor = -factor;
  }
  return 4.0 * sum;
}

// ============================================================================
// V1: factor is shared and carries a dependence across iterations.
// ============================================================================
static double pi_v1_shared_dep(long n, int thread_count) {
  double sum = 0.0;
  double factor = 1.0;

#pragma omp parallel for num_threads(thread_count) reduction(+ : sum)
  for (long k = 0; k < n; k++) {
    sum += factor / (double)(2 * k + 1);
    factor = -factor;
  }
  return 4.0 * sum;
}

// ============================================================================
// V2: firstprivate gives every thread its own copy initialised to 1.0, so the
// data race disappears. The dependence does not: a chunk starting at an odd k
// must start with factor = -1.0.
// ============================================================================
static double pi_v2_firstprivate_dep(long n, int thread_count) {
  double sum = 0.0;
  double factor = 1.0;

#pragma omp parallel for num_threads(thread_count) reduction(+ : sum) \
    firstprivate(factor)
  for (long k = 0; k < n; k++) {
    sum += factor / (double)(2 * k + 1);
    factor = -factor;
  }
  return 4.0 * sum;
}

// ============================================================================
// V3: factor is now computed from k, so the dependence is gone. The variable
// is still shared, so threads keep overwriting each other's sign.
// ============================================================================
static double pi_v3_shared_race(long n, int thread_count) {
  double sum = 0.0;
  volatile double factor = 1.0;

#pragma omp parallel for num_threads(thread_count) reduction(+ : sum)
  for (long k = 0; k < n; k++) {
    factor = (k % 2 == 0) ? 1.0 : -1.0;
    sum += factor / (double)(2 * k + 1);
  }
  return 4.0 * sum;
}

// ============================================================================
// V4: private removes the sharing. The initialiser below is never copied into
// the private replicas, which is why factor must be assigned before every use.
// ============================================================================
static double pi_v4_private(long n, int thread_count) {
  double sum = 0.0;
  double factor = 0.0;

#pragma omp parallel for num_threads(thread_count) \
    reduction(+ : sum) private(factor)
  for (long k = 0; k < n; k++) {
    factor = (k % 2 == 0) ? 1.0 : -1.0;
    sum += factor / (double)(2 * k + 1);
  }
  return 4.0 * sum;
}

// ============================================================================
// V5: merge two consecutive terms algebraically. No modulo, no branch and no
// sign variable, which leaves a loop the compiler can vectorise.
// ============================================================================
static double pi_v5_math_unroll(long n, int thread_count) {
  double sum = 0.0;
  long pairs = n / 2;

#pragma omp parallel for num_threads(thread_count) reduction(+ : sum)
  for (long k = 0; k < pairs; k++) {
    sum += 1.0 / (double)(4 * k + 1) - 1.0 / (double)(4 * k + 3);
  }

  // An odd n leaves one positive term without a partner.
  if (n % 2 != 0) {
    sum += 1.0 / (double)(2 * n - 1);
  }
  return 4.0 * sum;
}

int main(int argc, char *argv[]) {
  if (argc != 3) {
    printf("Usage: %s <n> <thread_count>\n", argv[0]);
    printf("Example: %s 100000001 4\n", argv[0]);
    return 1;
  }

  long n = strtol(argv[1], NULL, 10);
  int thread_count = (int)strtol(argv[2], NULL, 10);

  if (n <= 0) {
    printf("Error: n must be > 0\n");
    return 1;
  }
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }

  printf("%s\n", BANNER);
  printf(" Lab 3 (part 2): Pi Estimation and Loop-Carried Dependence\n");
  printf(" n: %ld | Threads: %d | Runs: %d\n", n, thread_count, NTIMES);
  printf(" _OPENMP: %d | Procs: %d\n", _OPENMP, omp_get_num_procs());
  printf("%s\n", BANNER);

  if (omp_get_num_procs() < 2 && n < 100000000L) {
    printf("\n[Warning] One processor and n < 1e8. V3 relies on preemption\n");
    printf("          inside a very short window and may report PASS here.\n");
    printf("          Increase n or use more cores to expose the race.\n");
  }
  if (n % 2 == 0) {
    printf("\n[Warning] n is even. V2 may return the correct value by\n");
    printf("          accident when every static chunk starts at an even\n");
    printf("          index. An odd n makes the defect reproducible.\n");
  }

  double start = 0.0;
  double t[6] = {0.0};
  double res[6] = {0.0};

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    res[0] = pi_serial(n);
    t[0] += get_time_ms() - start;
  }

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    res[1] = pi_v1_shared_dep(n, thread_count);
    t[1] += get_time_ms() - start;
  }

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    res[2] = pi_v2_firstprivate_dep(n, thread_count);
    t[2] += get_time_ms() - start;
  }

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    res[3] = pi_v3_shared_race(n, thread_count);
    t[3] += get_time_ms() - start;
  }

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    res[4] = pi_v4_private(n, thread_count);
    t[4] += get_time_ms() - start;
  }

  for (int r = 0; r < NTIMES; r++) {
    start = get_time_ms();
    res[5] = pi_v5_math_unroll(n, thread_count);
    t[5] += get_time_ms() - start;
  }

  for (int i = 0; i < 6; i++) {
    t[i] /= NTIMES;
  }

  print_table_header();
  print_row("V0: Serial Baseline", t[0], t[0], -1);
  print_row("V1: Shared + Dependence", t[1], t[0],
            check_diff(&res[0], &res[1], 1, TOL));
  print_row("V2: Firstprivate + Dep.", t[2], t[0],
            check_diff(&res[0], &res[2], 1, TOL));
  print_row("V3: Shared Variable Race", t[3], t[0],
            check_diff(&res[0], &res[3], 1, TOL));
  print_row("V4: Private + Reduction", t[4], t[0],
            check_diff(&res[0], &res[4], 1, TOL));
  print_row("V5: Math Unrolling", t[5], t[0],
            check_diff(&res[0], &res[5], 1, TOL));
  printf("%s\n", LINE);

  printf("\nReference pi (V0): %.15f\n", res[0]);
  printf("V1 %.15f  V2 %.15f  V3 %.15f\n", res[1], res[2], res[3]);
  printf("V4 %.15f  V5 %.15f\n", res[4], res[5]);

  return 0;
}

## 10. 编译与运行

### 10.1 编译

In [ ]:
bin_pi = compile_c('omp_pi_dependency.c')

### 10.2 运行

参数为项数与线程数。项数取 `100000001`（**奇数**），确保 8.3 节所述的块起始奇偶条件成立。

预期：V1、V2 显示 FAIL；V3 在多核平台上显示 FAIL，单核平台上因 $n$ 已达 $10^8$ 量级通常也显示 FAIL；V4、V5 显示 PASS。

In [ ]:
out_pi = run_c(bin_pi, 100000001, 4)
rows_pi = parse_table(out_pi)
print()
print('校验结论汇总：')
for name, ms, sp, chk in rows_pi:
    mark = {'PASS': '✅', 'FAIL': '❌'}.get(chk, '  ')
    print('  %s %-28s %8.3f ms  %5.2fx' % (mark, name, ms, sp))

### 10.3 确定性对照：同一版本重复运行

V2 的错误来自**依赖**，其结果在各次运行之间应当保持一致；V3 的错误来自**竞争**，其结果应当在运行之间波动（多核平台上）。下面连续运行五次，提取两者的数值加以对照。

In [ ]:
import re

pat = re.compile(r'V1 ([\d.]+)\s+V2 ([\d.]+)\s+V3 ([\d.]+)')
print('%-6s %-20s %-20s %-20s' % ('run', 'V1 (依赖+竞争)', 'V2 (依赖)', 'V3 (竞争)'))
for trial in range(5):
    out = subprocess.run([bin_pi, '100000001', '4'],
                         capture_output=True, text=True).stdout
    m = pat.search(out)
    if m:
        print('%-6d %-20s %-20s %-20s'
              % (trial + 1, m.group(1), m.group(2), m.group(3)))

print()
print('若 V2 五次结果一致而 V3 存在波动，即验证了「依赖是确定性错误、')
print('竞争是非确定性错误」这一区分。归约的合并次序可能引起末位差异，')
print('比较时以打印位数为准；单核平台上 V3 亦可能保持稳定。')

## 11. 结果可视化

红色柱表示该版本**校验未通过**。需注意：校验未通过的版本，其耗时数据不具备参考价值。将其一并绘出，是为了说明正确性与性能是两个相互独立的维度，且正确性应优先于性能。

In [ ]:
plot_speedup(rows_pi,
             'Lab 3: Pi Estimation (n = 1e8+1, 4 threads)',
             figsize=(11, 5))

## 12. 结果分析

> 以下结论针对**趋势规律**，具体数值请以自己实验平台上的实测结果为准。

**① 三个错误版本均可无告警通过编译**

V1、V2、V3 在 `-Wall -Wextra` 下没有产生任何告警，运行时也不会崩溃。这印证了 1.4 节的结论：编译器不对被分担的循环进行依赖分析，并行正确性需由程序员自行保证。

**② V1 与 V2 可能给出完全相同的错误值**

若实测中两者数值一致，说明 `-O3` 把 V1 中共享的 `factor` 提升进了各线程的私有寄存器，其实际行为退化成了 V2。

这一现象有两层教学价值：

- 它说明**优化会改变并发程序的可观测行为**。在 `-O0` 下重新编译 V1，很可能得到与 V2 不同的结果。
- 它解释了为什么并发缺陷「在 Debug 版本正常、Release 版本出错」这类现象如此常见。

**③ V2 的错误是确定的，V3 的错误是随机的**

10.3 节的重复运行对照给出了直接证据。这一区分在实际调试中很有价值：

| 观察到的现象 | 应当怀疑 |
|---|---|
| 每次运行结果完全相同，但与串行不符 | 循环携带依赖 |
| 每次运行结果略有不同 | 数据竞争 |
| 线程数为 1 时正确，多线程时错误 | 两者皆有可能 |

**④ 缩小 $n$ 会使 V3 的错误消失**

把 $n$ 从 $10^8$ 降到 $2\times10^7$ 后，单核平台上的 V3 往往报告 PASS。这不是程序变对了，而是竞争窗口太窄、抢占未能命中。由此可得出一条工程原则：**测试所用的核心数与问题规模，都不应低于实际部署环境**，否则并发缺陷可能在部署之后才被发现。

**⑤ V5 通常是最快的版本**

V5 的循环次数只有其余版本的一半，且循环体内没有取模与分支，编译器更容易实施向量化。这说明：**从算法层面进行改写，往往比在并行框架内做局部修补更为有效**。

**⑥ 关于 V4 与 V5 的结果并不等于 $\pi$**

两者与 $\pi$ 的差距约在 $10^{-8}$ 量级，这是级数的截断误差，属于算法的数学性质，与并行化无关。校验以串行结果为参照，正是为了把这一项排除在外。

## 13. 🔧 动手练习

**练习 1**　依次以 `-O0`、`-O1`、`-O2`、`-O3` 重新编译案例二，找出 V1 与 V2 数值由「不同」转为「相同」的优化级别分界，并解释原因。

**练习 2**　把 `n` 改为**偶数**（例如 `100000000`），线程数取 4，观察 V2 的校验结论。请解释为何一个逻辑错误的程序会给出正确结果。

**练习 3**　把案例一的 `n` 改为 46，线程数取 2、3、5、7，记录首个错误下标，验证它是否总是落在某个线程块的起始位置。

**练习 4**　固定线程数为 4，把 `n` 依次取 $10^6$、$10^7$、$10^8$，记录 V3 的校验结论，找出该平台上竞争开始稳定复现的规模阈值。

**练习 5**　为 V4 增加一个「倒序执行」的对照版本（循环从 `n-1` 递减到 `0`），验证 2.1 节所述的倒序自检方法。

**练习 6**（进阶）　把 V4 中的 `private(factor)` 删去，改为把 `double factor` 声明在循环体内部，确认结果依然正确，并说明这两种写法在语义上的关系。

**练习 7**（进阶）　尝试为 V5 加上 `#pragma omp simd`，观察是否有额外收益。结合第三章的知识解释所观察到的现象。

### 13.1 练习 1 的参考实现：不同优化级别下的 V1 与 V2

In [ ]:
for opt in ('-O0', '-O1', '-O2', '-O3'):
    binary = os.path.join(SRC_DIR, 'omp_pi_dependency_%s' % opt.lstrip('-'))
    subprocess.run(['gcc', opt, '-fopenmp', '-Wall', '-Wextra',
                    '-o', binary,
                    os.path.join(SRC_DIR, 'omp_pi_dependency.c'),
                    '-lm'], check=True)
    out = subprocess.run([binary, '100000001', '4'],
                         capture_output=True, text=True).stdout
    m = re.search(r'V1 ([\d.]+)\s+V2 ([\d.]+)', out)
    if m:
        same = '相同' if m.group(1) == m.group(2) else '不同'
        print('%-4s  V1 = %s   V2 = %s   → %s'
              % (opt, m.group(1), m.group(2), same))

### 13.2 练习 2 的参考实现：奇偶 `n` 对 V2 的影响

In [ ]:
for n in (100000000, 100000001):
    out = subprocess.run([bin_pi, str(n), '4'],
                         capture_output=True, text=True).stdout
    rows = parse_table(out)
    v2 = [r for r in rows if r[0].startswith('V2')]
    parity = '偶数' if n % 2 == 0 else '奇数'
    if v2:
        print('n = %-11d (%s)  →  V2 校验：%s' % (n, parity, v2[0][3]))
print()
print('提示：n 为偶数且每块长度亦为偶数时，所有块的起始下标均为偶数，')
print('      firstprivate 的初值 +1 恰好正确，错误因而被掩盖。')

### 13.3 练习 4 的参考实现：V3 的规模阈值

In [ ]:
for n in (1000000, 10000000, 100000000):
    out = subprocess.run([bin_pi, str(n + 1), '4'],
                         capture_output=True, text=True).stdout
    rows = parse_table(out)
    v3 = [r for r in rows if r[0].startswith('V3')]
    if v3:
        print('n = %-11d  →  V3 校验：%s' % (n + 1, v3[0][3]))
print()
print('阈值随平台而变。核心数越多，越小的 n 即可复现竞争。')

## 14. 🤔 思考题

**思考题 1**　案例一中，若把线程数设为 1，程序结果正确。能否由此断定该程序「在单线程下是对的，只是不支持多线程」？这种说法的问题在哪里？

**思考题 2**　下面这个循环是否存在循环携带依赖？若有，属于哪一种？能否消除？

```c
for (int i = 0; i < n - 1; i++) {
  a[i] = a[i + 1] + b[i];
}
```

**思考题 3**　V3 使用 `volatile` 是为了让竞争可观测。若把 V1 中的 `factor` 也加上 `volatile`，V1 的结果会如何变化？它还会与 V2 相同吗？

**思考题 4**　实验二指出 `reduction` 相当于「私有副本 + 合并」。既然如此，为什么本实验不能简单地对 `factor` 也加一个 `reduction`？`reduction` 适用于什么样的变量？

**思考题 5**　V5 把 `n % 2 != 0` 的落单项放在循环**之外**处理。若把这个判断放进循环体内，会带来什么后果？（提示：从分支预测与向量化两个角度考虑。）

**思考题 6**（综合）　假设需要维护一段遗留代码，其中有一个上千行的循环，无法逐行分析其依赖关系。请设计一套**实验性**的检测方案，在不阅读全部代码的前提下判断该循环是否可以并行。（提示：结合 2.1 节的倒序自检与本实验的重复运行对照。）

## 15. 📌 本实验小结

| 概念 | 要点 |
|---|---|
| 循环携带依赖 | 第 $i$ 次迭代读写第 $j$ 次迭代所写的数据 |
| 三种形式 | 流依赖（真）、反依赖与输出依赖（伪，可消除） |
| 编译器的职责边界 | `#pragma omp for` 是命令，编译器不做依赖分析 |
| 可消除的依赖 | 归纳变量、符号递推 —— 改写为闭式表达 |
| 不可消除的依赖 | 真实数据流，只能更换算法 |
| 依赖 vs 竞争 | 前者确定性错误，后者非确定性错误，需分别修复 |
| `private` vs `firstprivate` | 后者复制初值，可把随机错误变为确定错误 |
| 寄存器提升 | `-O3` 可能使共享变量的竞争不可观测 |
| `volatile` | 仅禁止优化，**不能**解决数据竞争 |

### 三条可直接使用的判断准则

1. **结果每次相同但与串行不符** → 怀疑循环携带依赖；
2. **结果每次不同** → 怀疑数据竞争；
3. **单线程正确、多线程出错** → 两者皆需排查，且优先检查数据环境。

### 与后续实验的衔接

至此，本章已经解决了并行程序的**正确性**问题：实验二给出数据环境规则，实验三给出依赖的识别与消除方法。从下一个实验起，讨论转向**性能**：

- 迭代之间的工作量若不相等，静态划分会造成部分线程空等。如何调整划分策略？→ 实验四
- 并行区域反复创建与销毁的代价有多大？→ 实验五
- 同步构造本身的开销如何度量与规避？→ 实验六